In [23]:
import pandas as pd
from datetime import timedelta

# === Read the CSV file ===
# Parse 'data_timestamp' as datetime to allow time-based operations
df = pd.read_csv('final_output_.csv', parse_dates=['data_timestamp'])

# === Get the earliest timestamp in the dataset (Hour Group 0 start) ===
start_hour_0 = df['data_timestamp'].min()
print("Start Hour 0:", start_hour_0)

# === Calculate data extraction time range ===
# Shift forward by 2 hours to define new extraction starting point
start_time = start_hour_0 + timedelta(hours=2)

# Define the end point: 5 hours after the new start time
end_time = start_time + timedelta(hours=5)

# === Print the intended extraction window ===
print("Start extracting from:", start_time)
print("End extracting at:", end_time)


Start Hour 0: 2025-05-01 19:53:44.127245
Start extracting from: 2025-05-01 21:53:44.127245
End extracting at: 2025-05-02 02:53:44.127245


In [25]:
import requests
import pandas as pd

API_KEY = 'beBybSi8daPgsTp5yx5cHtHpYcrjp5Jq'

def fetch_polygon_data(pair, start_time, end_time):
    """
    Fetch 6-minute interval price data from Polygon.io for a given currency pair.
    
    Parameters:
        pair (str): Currency pair symbol (e.g., 'USDJPY')
        start_time (datetime): Start of the time range
        end_time (datetime): End of the time range
    
    Returns:
        pd.DataFrame: DataFrame with timestamp and closing price columns
    """
    symbol = f'C:{pair}'  # Polygon uses 'C:' prefix for forex pairs
    url = f"https://api.polygon.io/v2/aggs/ticker/{symbol}/range/6/minute/{start_time.strftime('%Y-%m-%d')}/{end_time.strftime('%Y-%m-%d')}"
    
    params = {
        "adjusted": "true",
        "sort": "asc",
        "limit": 5000,
        "apiKey": API_KEY
    }
    
    response = requests.get(url, params=params)
    
    # Handle potential request failure
    if response.status_code != 200:
        print(f"Error fetching {pair}: {response.status_code}")
        return pd.DataFrame()

    data = response.json().get('results', [])

    # Convert to DataFrame and filter time range precisely
    df = pd.DataFrame(data)
    if df.empty:
        return df
    
    df['timestamp'] = pd.to_datetime(df['t'], unit='ms')
    df = df[(df['timestamp'] >= start_time) & (df['timestamp'] < end_time)]
    df = df[['timestamp', 'c']].rename(columns={'c': pair})
    
    return df


In [27]:
# Fetch 6-minute data for USDGBP and USDJPY
usdgbp_df = fetch_polygon_data("USDGBP", start_time, end_time)
usdjpy_df = fetch_polygon_data("USDJPY", start_time, end_time)

# Optional: Normalize USDJPY to similar scale (Polygon returns JPY in ~100x scale)
usdjpy_df["USDJPY"] = usdjpy_df["USDJPY"] / 100

# Merge the two price series on timestamp
# This keeps only matching time points (inner join)
price_df = pd.merge(usdgbp_df, usdjpy_df, on='timestamp', how='inner')

# Display merged DataFrame
print(price_df)


             timestamp   USDGBP   USDJPY
0  2025-05-01 21:54:00  0.75310  1.45321
1  2025-05-01 22:00:00  0.75294  1.45331
2  2025-05-01 22:06:00  0.75302  1.45382
3  2025-05-01 22:12:00  0.75298  1.45352
4  2025-05-01 22:18:00  0.75281  1.45320
5  2025-05-01 22:24:00  0.75279  1.45328
6  2025-05-01 22:30:00  0.75269  1.45291
7  2025-05-01 22:36:00  0.75266  1.45294
8  2025-05-01 22:42:00  0.75267  1.45256
9  2025-05-01 22:48:00  0.75272  1.45316
10 2025-05-01 22:54:00  0.75271  1.45331
11 2025-05-01 23:00:00  0.75288  1.45439
12 2025-05-01 23:06:00  0.75286  1.45434
13 2025-05-01 23:12:00  0.75286  1.45463
14 2025-05-01 23:18:00  0.75277  1.45467
15 2025-05-01 23:24:00  0.75276  1.45506
16 2025-05-01 23:30:00  0.75279  1.45501
17 2025-05-01 23:36:00  0.75288  1.45550
18 2025-05-01 23:42:00  0.75277  1.45508
19 2025-05-01 23:48:00  0.75267  1.45427
20 2025-05-01 23:54:00  0.75265  1.45405
21 2025-05-02 00:00:00  0.75277  1.45518
22 2025-05-02 00:06:00  0.75294  1.45533
23 2025-05-02 00

In [75]:
from sklearn.linear_model import LinearRegression
import numpy as np

# === Compute linear slope of a price series (trend direction) ===
def compute_slope(price_series):
    X = np.arange(len(price_series)).reshape(-1, 1)
    y = np.array(price_series).reshape(-1, 1)
    model = LinearRegression().fit(X, y)
    return model.coef_[0][0]  # Return the slope


# === Slope-based Long/Short Trading Strategy ===
def trading_strategy(price_df, open_idx=20, close_idx=29):
    usdgbp = 'USDGBP'
    usdjpy = 'USDJPY'

    # Compute trend (slope) from first 20 observations
    slope_gbp = compute_slope(price_df[usdgbp].values[:20])
    slope_jpy = compute_slope(price_df[usdjpy].values[:20])

    # Get entry and exit prices for both pairs
    long_open_usdgbp = price_df.iloc[open_idx][usdgbp]
    long_close_usdgbp = price_df.iloc[close_idx][usdgbp]
    long_open_usdjpy = price_df.iloc[open_idx][usdjpy]
    long_close_usdjpy = price_df.iloc[close_idx][usdjpy]

    # Show trend direction
    print(f"📈 Slope USDGBP: {slope_gbp:.6f}")
    print(f"📈 Slope USDJPY: {slope_jpy:.6f}")

    # === Case 1: Both trending up → Long both proportionally ===
    if slope_gbp > 0 and slope_jpy > 0:
        part_usdgbp = slope_gbp / (slope_gbp + slope_jpy)
        part_usdjpy = 1 - part_usdgbp
        print(f"🟢 Both trending up → Long USDGBP {part_usdgbp:.2%}, Long USDJPY {part_usdjpy:.2%}")
        gbp_pl = 100 * part_usdgbp * (long_close_usdgbp - long_open_usdgbp) / long_open_usdgbp
        jpy_pl = 100 * part_usdjpy * (long_close_usdjpy - long_open_usdjpy) / long_open_usdjpy
        pl = gbp_pl + jpy_pl
        print(f"USDGBP P/L = ${gbp_pl:.4f}, USDJPY P/L = ${jpy_pl:.4f}")

    # === Case 2: Both trending down → Short both proportionally ===
    elif slope_gbp < 0 and slope_jpy < 0:
        part_usdgbp = abs(slope_gbp) / (abs(slope_gbp) + abs(slope_jpy))
        part_usdjpy = 1 - part_usdgbp
        print(f"🔻 Both trending down → Short USDGBP {part_usdgbp:.2%}, Short USDJPY {part_usdjpy:.2%}")
        gbp_pl = 100 * part_usdgbp * (long_open_usdgbp - long_close_usdgbp) / long_open_usdgbp
        jpy_pl = 100 * part_usdjpy * (long_open_usdjpy - long_close_usdjpy) / long_open_usdjpy
        pl = gbp_pl + jpy_pl
        print(f"USDGBP P/L = ${gbp_pl:.4f}, USDJPY P/L = ${jpy_pl:.4f}")

    # === Case 3: Mixed signal — GBP up, JPY down → Long GBP, Short JPY ===
    elif slope_gbp > 0 and slope_jpy < 0:
        print("📊 Mixed → Long USDGBP 100%, Short USDJPY 100%")
        gbp_pl = 100 * (long_close_usdgbp - long_open_usdgbp) / long_open_usdgbp
        jpy_pl = 100 * (long_open_usdjpy - long_close_usdjpy) / long_open_usdjpy
        pl = gbp_pl + jpy_pl
        print(f"USDGBP P/L = ${gbp_pl:.4f}, USDJPY P/L = ${jpy_pl:.4f}")

    # === Case 4: Mixed signal — GBP down, JPY up → Long JPY, Short GBP ===
    elif slope_gbp < 0 and slope_jpy > 0:
        print("📊 Mixed → Long USDJPY 100%, Short USDGBP 100%")
        gbp_pl = 100 * (long_open_usdgbp - long_close_usdgbp) / long_open_usdgbp
        jpy_pl = 100 * (long_close_usdjpy - long_open_usdjpy) / long_open_usdjpy
        pl = gbp_pl + jpy_pl
        print(f"USDGBP P/L = ${gbp_pl:.4f}, USDJPY P/L = ${jpy_pl:.4f}")

    # === Case 5: No slope or flat ===
    else:
        pl = 0
        print("⚠️ No clear signal. No position taken.")

    print(f"💰 Total P/L = ${pl:.4f}\n")
    return float(pl)


In [79]:
# Hour 5
slope_gbp_h5 = compute_slope(price_df['USDGBP'].values[0:20])
slope_jpy_h5 = compute_slope(price_df['USDJPY'].values[0:20])
pl_hour5 = trading_strategy(price_df, open_idx=20, close_idx=29)

📈 Slope USDGBP: -0.000010
📈 Slope USDJPY: 0.000115
📊 Mixed → Long USDJPY 100%, Short USDGBP 100%
USDGBP P/L = $-0.0771, USDJPY P/L = $0.3205
💰 Total P/L = $0.2434



In [81]:
# Hour 6
slope_gbp_h6 = compute_slope(price_df['USDGBP'].values[10:30])
slope_jpy_h6 = compute_slope(price_df['USDJPY'].values[10:30])
pl_hour6 = trading_strategy(price_df.iloc[10:], open_idx=20, close_idx=29)

📈 Slope USDGBP: 0.000020
📈 Slope USDJPY: 0.000278
🟢 Both trending up → Long USDGBP 6.63%, Long USDJPY 93.37%
USDGBP P/L = $-0.0048, USDJPY P/L = $-0.2496
💰 Total P/L = $-0.2545



In [83]:
# Hour 7
slope_gbp_h7 = compute_slope(price_df['USDGBP'].values[20:40])
slope_jpy_h7 = compute_slope(price_df['USDJPY'].values[20:40])
pl_hour7 = trading_strategy(price_df.iloc[20:], open_idx=20, close_idx=29)


📈 Slope USDGBP: -0.000006
📈 Slope USDJPY: -0.000039
🔻 Both trending down → Short USDGBP 12.89%, Short USDJPY 87.11%
USDGBP P/L = $0.0031, USDJPY P/L = $-0.0078
💰 Total P/L = $-0.0047



In [89]:
# Total Return
total_pl = pl_hour5 + pl_hour6 + pl_hour7
print(f"🎯 Total P/L over 3 hours = ${total_pl:.4f}")

🎯 Total P/L over 3 hours = $-0.0158
